In [1]:
import os, collections
import datetime
from datetime import date
import pandas as pd
import numpy as np
import re

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.sequence import pad_sequences


In [2]:
import tensorflow
print(tensorflow.__version__)

2.4.1


In [3]:
def read_data_from_file(filename, data_dict):

    with open(filename) as fp:
        line = fp.readline()
        while line:
            bo, ch, ve, text = tuple(line.strip().split('\t'))
            words = text.split()
            for w in words:  
                # in the output data, composite placenames have a '_', which cannot be found in the input data
                words_split = w.split('_')               
                for word_split in words_split:
                    data_dict[bo].append(word_split)
        
            line = fp.readline()
            
    return data_dict

In [4]:
input_file = 'data/t-in_voc'
#input_file = 't-in_voc'
input_data = collections.defaultdict(list)

output_file = 'data/t-out'
#output_file = 't-out'
output_data = collections.defaultdict(list)

input_data = read_data_from_file(input_file, input_data)
output_data = read_data_from_file(output_file, output_data)

In [5]:
print(len(input_data['Gen']))
print(len(output_data['Gen']))

20611
20611


In [6]:
# The reduction consists of removing the left-most marker from all
# the doubly marked prefixes and the redundant colon of the vowel
# pattern mark.

prefixes = ['!', ']', '@']

def mc_reduce(s):
   for c in prefixes:
      s = re.sub(f'{c}([^{c}]*{c})', r'\1', s)
   return s.replace(':', '')

In [7]:
def make_in_sequences(data_dict, sequence_length):
    
    all_sequences = []
    for words_list in data_dict.values():

        for w in range(len(words_list) - sequence_length + 1):
    
            seq = ' '.join([words_list[ind] for ind in list(range(w, w + sequence_length))])
        
            all_sequences.append(seq)
        
    return all_sequences

In [8]:
def make_out_sequences(data_dict, sequence_length):
    
    all_sequences = []
    for words_list in data_dict.values():

        for w in range(len(words_list) - sequence_length + 1):
    
            seq = ' '.join([words_list[ind] for ind in list(range(w, w + sequence_length))])
        
            seq = mc_reduce(seq)
            all_sequences.append(seq)
        
    return all_sequences

In [9]:
sequence_length = 1

all_in_seqs = make_in_sequences(input_data, sequence_length)
all_out_seqs = make_out_sequences(output_data, sequence_length)


In [10]:
def prepare_train_data(input_data, output_data):

    input_seqs = []
    output_seqs = []
    input_chars = set()
    output_chars = set()

    # iterate over all the books
    for seq in range(len(input_data)): 
      
        #if len(output_data[seq]) > 40:
        #  continue
          
        if "*" in input_data[seq]: # cases of ketiv/qere are complicated, just skip them!
          continue

        input_list = list(input_data[seq])

        output_list = list(output_data[seq])
        #output_list = ['\t'] + output_list + ['\n']

        input_seqs.append(input_list)
        output_seqs.append(output_list)

        for input_ch in input_list:
            input_chars.add(input_ch)
        
        for output_ch in output_list:
            output_chars.add(output_ch)
                
    
    input_chars = sorted(list(input_chars))
    output_chars = sorted(list(output_chars))
    
    max_len_input = max([len(seq) for seq in input_seqs])
    max_len_output = max([len(seq) for seq in output_seqs])
    
    # shuffle the data. The model will get the data in small batches, it is preferable if the batches are more or less homogeneous
    # of course the inputs and outputs have to be shuffled identically
    # input_seqs, output_seqs = shuffle(input_seqs, output_seqs)
    
    return input_seqs, output_seqs, input_chars, output_chars, max_len_input, max_len_output

In [11]:
def create_dicts(input_voc, output_voc):
    
    # these dicts map the input sequences
    input_idx2char = {}
    input_char2idx = {}
    
    input_idx2char[0] = 'PAD'
    input_char2idx['PAD'] = 0

    for k, v in enumerate(input_voc):
        input_idx2char[k + 1] = v
        input_char2idx[v] = k + 1
     
    # and these dicts map the output sequences of parts of speech~
    output_idx2char = {}
    output_char2idx = {}
    
    output_idx2char[0] = 'PAD'
    output_char2idx['PAD'] = 0
    
    for k, v in enumerate(output_voc):
        output_idx2char[k + 1] = v
        output_char2idx[v] = k + 1
        
    return input_idx2char, input_char2idx, output_idx2char, output_char2idx

In [12]:
def pad(x, length=None):

    padded_sequences = pad_sequences(x, length, padding='post')
    
    return padded_sequences

In [13]:
input_seqs, output_seqs, input_chars, output_chars, max_len_input, max_len_output = prepare_train_data(all_in_seqs, all_out_seqs)
print(len(input_seqs))

299488


In [29]:
input_seqs[0]

['B', '.', ':', 'R', ';', '>', 'C', 'I', 'J', 'T']

In [14]:
input_idx2char, input_char2idx, output_idx2char, output_char2idx = create_dicts(input_chars, output_chars)

In [15]:
input_seqs_num = []
output_seqs_num = []

for seq in input_seqs:
    input_seqs_num.append([input_char2idx[char] for char in seq])
    
for seq in output_seqs:
    output_seqs_num.append([output_char2idx[char] for char in seq])

In [16]:
input_seqs_num = pad(input_seqs_num)
output_seqs_num = pad(output_seqs_num)

# sparse_categorical_crossentropy function requires the labels to be in 3 dims
output_seqs_num = output_seqs_num.reshape(*output_seqs_num.shape, 1)

In [17]:
max_len_input = max([len(seq) for seq in input_seqs_num])
max_len_output = max([len(seq) for seq in output_seqs_num])
max_len_output

24

In [18]:
prepared_train_x = pad(input_seqs_num, max_len_output)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(prepared_train_x, output_seqs_num, test_size=0.1, random_state=42)


In [20]:
# Write preprocessed data to files
import os
os.mkdir('data/preprocessed')

In [21]:
X_train.shape

(269539, 24)

In [22]:
X_train[0]

array([ 8,  1, 11, 20,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0], dtype=int32)

In [25]:
np.save('data/preprocessed/X_train.npy', X_train)
np.save('data/preprocessed/X_test.npy', X_test)
np.save('data/preprocessed/y_train.npy', y_train)
np.save('data/preprocessed/y_test.npy', y_test)

In [43]:
import pickle
with open('data/preprocessed/dicts.pkl', 'wb') as fout:
    pickle.dump({
        'input_idx2char': input_idx2char,
        'input_char2idx': input_char2idx,
        'output_idx2char': output_idx2char,
        'output_char2idx': output_char2idx
    }, fout)

In [34]:
input_train, input_test, output_train, output_test = train_test_split(input_seqs, output_seqs, test_size=0.1, random_state=42)


In [37]:
with open('data/preprocessed/input_seqs_train.txt', 'w') as fout:
    for seq in input_train:
        fout.write(''.join(seq)+'\n')
        
with open('data/preprocessed/output_seqs_train.txt', 'w') as fout:
    for seq in output_train:
        fout.write(''.join(seq)+'\n')
        
with open('data/preprocessed/input_seqs_test.txt', 'w') as fout:
    for seq in input_test:
        fout.write(''.join(seq)+'\n')
        
with open('data/preprocessed/output_seqs_test.txt', 'w') as fout:
    for seq in output_test:
        fout.write(''.join(seq)+'\n')

In [40]:
pickle.dump?